# Зчитування та очищення даних
# Звантажити та відкрити датасет: Individual Household Electric Power Consumption Dataset. Здійснити data cleaning.

In [2]:
import pandas as pd
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
csv_path = "household_power_consumption.txt"

if not os.path.exists(csv_path):
    print("Завантажуємо датасет...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Розпаковано!")

print("Зчитуємо та очищуємо дані...")
df_power = pd.read_csv(csv_path, sep=';', na_values=['?'], low_memory=False)
df_power = df_power.dropna()
df_power['Datetime'] = pd.to_datetime(df_power['Date'] + ' ' + df_power['Time'], format='%d/%m/%Y %H:%M:%S')

print("Готово! Перші 5 рядків:")
display(df_power.head())

Завантажуємо датасет (це може зайняти пару хвилин)...
Розпаковано!
Зчитуємо та очищуємо дані...
Готово! Перші 5 рядків:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


# Завдання 2: Обрати записи, у яких сила струму лежить в межах 19-20 А, 
# і пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [3]:
# перетворюю потрібні стовпці у числовий формат
df_power['Global_intensity'] = pd.to_numeric(df_power['Global_intensity'], errors='coerce')
df_power['Sub_metering_2'] = pd.to_numeric(df_power['Sub_metering_2'], errors='coerce') # пральна машина та холодильник
df_power['Sub_metering_3'] = pd.to_numeric(df_power['Sub_metering_3'], errors='coerce') # бойлер та кондиціонер

def filter_intensity_appliances(df):
    # умова
    mask_intensity = (df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)
    mask_appliances = df['Sub_metering_2'] > df['Sub_metering_3']
    return df[mask_intensity & mask_appliances]

res_task2 = filter_intensity_appliances(df_power)
print(f"Знайдено записів: {len(res_task2)}")
display(res_task2.head())

print("Зачекайте...")
%timeit filter_intensity_appliances(df_power)

Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


Зачекайте...
10.1 ms ± 173 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


# Завдання 3: Випадковим чином обрати 500 000 записів та обчислити середні величини 3-х груп споживання

In [5]:
# перетворюю 2 та 3 групу у числовий формат
df_power['Sub_metering_1'] = pd.to_numeric(df_power['Sub_metering_1'], errors='coerce')

def random_sample_means(df):
    # обираю 500000
    sample_df = df.sample(n=500000, replace=False)
    
    # середнє значення
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

res_task3 = random_sample_means(df_power)
print("Середні величини споживання:")
print(res_task3)

# час виконання 
print("\nВимірюю час...")
%timeit random_sample_means(df_power)

Середні величини споживання:
Sub_metering_1    1.134768
Sub_metering_2    1.301252
Sub_metering_3    6.461758
dtype: float64

Вимірюю час...
203 ms ± 39.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# Завдання 4: Складна вибірка за часом (після 18:00), потужністю (> 6 кВт) та індексами

In [6]:
def filter_complex(df):
    # умови
    mask_time_power = (df['Datetime'].dt.hour >= 18) & (df['Global_active_power'] > 6.0)
    filtered_1 = df[mask_time_power]
    
    # залишаю ті, де група 2 більша за групу 1 та групу 3
    mask_group2_max = (filtered_1['Sub_metering_2'] > filtered_1['Sub_metering_1']) & \
                      (filtered_1['Sub_metering_2'] > filtered_1['Sub_metering_3'])
    filtered_2 = filtered_1[mask_group2_max]
    
    # ділю таблицю навпіл
    half_index = len(filtered_2) // 2
    first_half = filtered_2.iloc[:half_index]
    second_half = filtered_2.iloc[half_index:]
    
    # кожен третій рядок з першої половини, кожен четвертий з другої
    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]
    
    # об'єдную результати
    return pd.concat([res_first, res_second])

# викликаю функцію та виводжу результат
res_task4 = filter_complex(df_power)
print(f"Знайдено фінальних записів: {len(res_task4)}")
display(res_task4.head())

# вимірення часу 
print("\nВимірюю час...")
%timeit filter_complex(df_power)

Знайдено фінальних записів: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00



Вимірюю час...
67.7 ms ± 2.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


# Завдання 5: Нормування, стандартизація, кореляція та One Hot Encoding

In [9]:

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# для швидкості розрахунків візьму наш останній відфільтрований датасет (res_task4) 
df_selected = res_task4.copy()

# нормування та стандартизація 
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

# застосую до стовпця потужності
df_selected['Power_Normalized'] = scaler_minmax.fit_transform(df_selected[['Global_active_power']])
df_selected['Power_Standardized'] = scaler_std.fit_transform(df_selected[['Global_active_power']])

print("Нормування та стандартизація виконані.")

# Кореляція Пірсона та Спірмена для потужності та сили струму 
pearson_corr = df_selected['Global_active_power'].corr(df_selected['Global_intensity'], method='pearson')
spearman_corr = df_selected['Global_active_power'].corr(df_selected['Global_intensity'], method='spearman')

print(f"Кореляція Пірсона: {pearson_corr:.4f}")
print(f"Кореляція Спірмена: {spearman_corr:.4f}")

# One Hot Encoding категоріального атрибута 
# спочатку створю категорію: витягнемо день тижня з дати
df_selected['Day_of_week'] = df_selected['Datetime'].dt.day_name()

# роблю One Hot Encoding
df_encoded = pd.get_dummies(df_selected, columns=['Day_of_week'])

print("\nОсь нові колонки днів тижня після One Hot Encoding:")
print([col for col in df_encoded.columns if 'Day_of_week' in col])

display(df_encoded.head())

Нормування та стандартизація виконані.
Кореляція Пірсона: 0.9943
Кореляція Спірмена: 0.9840

Ось нові колонки днів тижня після One Hot Encoding:
['Day_of_week_Friday', 'Day_of_week_Monday', 'Day_of_week_Saturday', 'Day_of_week_Sunday', 'Day_of_week_Thursday', 'Day_of_week_Tuesday', 'Day_of_week_Wednesday']


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Power_Normalized,Power_Standardized,Day_of_week_Friday,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00,0.010331,-1.030291,False,False,True,False,False,False,False
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00,0.065433,-0.708381,False,False,True,False,False,False,False
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00,0.082221,-0.610299,False,False,False,False,True,False,False
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00,0.448558,1.529900,False,False,False,False,True,False,False
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00,0.263883,0.450999,False,False,False,False,True,False,False
